# Phase 5: LLM + RAG
**LLM:** Ollama (local) — llama3.2:3b

**Vector store:** ChromaDB

**Orchestration:** LangChain

**Goal:** Generate automated insights and enable natural language querying of waste data

## 1. Setup Ollama Connection
Connect to local Ollama server and verify model is accessible.

Ollama runs as a local REST API on port 11434 by default.

In [1]:
import requests
import json

OLLAMA_URL   = 'http://localhost:11434'
OLLAMA_MODEL = 'llama3.2:3b'

def ollama_generate(prompt, model=OLLAMA_MODEL, max_tokens=500):
    """Send prompt to Ollama and return response text."""
    response = requests.post(
        f'{OLLAMA_URL}/api/generate',
        json={
            'model'  : model,
            'prompt' : prompt,
            'stream' : False,
            'options': {'num_predict': max_tokens}
        }
    )
    return response.json()['response']

# Test basic call
response = ollama_generate('What is industrial waste incineration?')
print('=== Ollama Test ===')
print(f'Model : {OLLAMA_MODEL}')
print(f'Response:\n{response}')

=== Ollama Test ===
Model : llama3.2:3b
Response:
Industrial waste incineration, also known as hazardous waste incineration, is a process of burning or combusting industrial waste to reduce its volume and generate energy. This method is often used for managing hazardous industrial waste that contains toxic substances, heavy metals, and other pollutants.

The incineration process involves heating the waste material to high temperatures (typically between 1,000°C to 2,000°C) in a controlled environment, usually a furnace or an incinerator. The heat breaks down the organic matter in the waste, converting it into ash, flue gas, and syngas (a mixture of carbon monoxide, hydrogen, and carbon dioxide).

Industrial waste incineration can offer several benefits, including:

1. **Reduced volume**: Incinerating industrial waste can significantly reduce its volume, making it easier to handle and dispose of.
2. **Energy generation**: The process generates heat energy, which can be used to power the

## 2. Verify Model Info
Check model details and confirm it is loaded correctly.

In [2]:
# Check available models
resp = requests.get(f'{OLLAMA_URL}/api/tags')
models = resp.json()['models']

print('Available Ollama models:')
for m in models:
    print(f"  {m['name']} — {round(m['size']/1e9, 1)} GB")

# Check model is ready
print(f'\nUsing model: {OLLAMA_MODEL}')
print('Status: Ready')

Available Ollama models:
  llama3.2:3b — 2.0 GB

Using model: llama3.2:3b
Status: Ready


## 3. Load Data from Cloud SQL
Query daily and weekly waste statistics from Cloud SQL to use as context for LLM summarization.

In [3]:
import pandas as pd
import psycopg2

DB_CONFIG = {
    'host'    : '34.104.146.13',
    'database': 'waste_intelligence',
    'user'    : 'postgres',
    'password': '12345678',
    'port'    : 5432
}

def query_db(sql, db_config):
    conn = psycopg2.connect(**db_config)
    df   = pd.read_sql(sql, conn)
    conn.close()
    return df

# Load daily summary
df_daily = query_db("""
    SELECT date, class_name, annotation_count, mean_bbox_area, image_count
    FROM daily_waste_summary
    ORDER BY date, class_name
""", DB_CONFIG)

# Load weekly summary
df_weekly = query_db("""
    SELECT week_start, class_name, annotation_count, image_count
    FROM weekly_waste_summary
    ORDER BY week_start, class_name
""", DB_CONFIG)

# Load anomaly results
df_anomaly = query_db("""
    SELECT date, is_anomaly, anomaly_score, annotation_count, explanation
    FROM anomaly_results
    WHERE is_anomaly = TRUE
    ORDER BY anomaly_score
""", DB_CONFIG)

print(f'Daily rows   : {len(df_daily)}')
print(f'Weekly rows  : {len(df_weekly)}')
print(f'Anomaly days : {len(df_anomaly)}')

C:\Users\bhumi\AppData\Local\Temp\ipykernel_29616\752173889.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql(sql, conn)


Daily rows   : 114
Weekly rows  : 37
Anomaly days : 4


## 4. Build Data-to-Text Context
Format structured data from Cloud SQL into a text context that can be injected into LLM prompts for summarization.

In [4]:
def build_weekly_context(df_weekly, df_anomaly, week_start=None):
    """
    Build text context from weekly waste statistics.
    If week_start is None, uses the most recent week.
    """
    if week_start is None:
        week_start = df_weekly['week_start'].max()

    df_week = df_weekly[df_weekly['week_start'] == week_start]

    # Format weekly stats
    lines = [f"Weekly Waste Report — Week of {str(week_start)[:10]}\n"]
    lines.append("Waste composition this week:")
    for _, row in df_week.iterrows():
        lines.append(f"  - {row['class_name']}: {int(row['annotation_count'])} annotations "
                    f"across {int(row['image_count'])} images")

    # Add anomaly context
    week_end = pd.Timestamp(week_start) + pd.Timedelta(days=6)
    df_anom_week = df_anomaly[
        (pd.to_datetime(df_anomaly['date']) >= pd.Timestamp(week_start)) &
        (pd.to_datetime(df_anomaly['date']) <= week_end)
    ]

    if len(df_anom_week) > 0:
        lines.append("\nAnomalous days detected this week:")
        for _, row in df_anom_week.iterrows():
            lines.append(f"  - {str(row['date'])[:10]}: {row['explanation']}")
    else:
        lines.append("\nNo anomalous days detected this week.")

    return '\n'.join(lines)

# Build context for most recent week
context = build_weekly_context(df_weekly, df_anomaly)
print('=== Data Context ===')
print(context)

=== Data Context ===
Weekly Waste Report — Week of 2024-09-30

Waste composition this week:
  - Metal: 300 annotations across 10 images
  - Mixed Waste: 292 annotations across 160 images
  - Paper-Cardboard: 817 annotations across 63 images
  - Plastic: 643 annotations across 82 images
  - Wood: 89 annotations across 18 images

Anomalous days detected this week:
  - 2024-09-30: Metal was unusually high at 300 (avg: 45)


## 5. Data-to-Text Pipeline
Send structured data context to Ollama and generate a human-readable weekly summary report.

In [5]:
def generate_weekly_report(context, model=OLLAMA_MODEL):
    """Generate weekly waste report from structured data context."""
    prompt = f"""You are an AI analyst for an industrial waste incineration facility in Japan.
Based on the following waste composition data, write a concise weekly operational report
in 3-4 sentences. Focus on key trends, notable changes, and any anomalies detected.
Use clear, professional language suitable for facility managers.

Data:
{context}

Weekly Report:"""

    return ollama_generate(prompt, max_tokens=300)

# Generate report for most recent week
report = generate_weekly_report(context)
print('=== Generated Weekly Report ===')
print(report)

=== Generated Weekly Report ===
**Weekly Operational Report**

This past week, the facility observed a notable increase in metal waste disposal, with 300 annotations detected on September 30th, exceeding the weekly average by nearly 6.7 times. This anomaly is being closely monitored to identify potential causes and prevent similar fluctuations in the future. Meanwhile, paper-cardboard and plastic waste compositions remained relatively stable, with no significant deviations from their typical ranges. No other anomalies were detected during this reporting period.


## 6. Generate Reports for All Weeks
Run Data-to-Text pipeline for all available weeks and store results in Cloud SQL llm_insights table.

In [6]:
weeks = sorted(df_weekly['week_start'].unique())
all_reports = []

print('Generating weekly reports...\n')
for week in weeks:
    context = build_weekly_context(df_weekly, df_anomaly, week_start=week)
    report  = generate_weekly_report(context)
    all_reports.append({
        'week_start': week,
        'context'   : context,
        'report'    : report
    })
    print(f'Week {str(week)[:10]}: Done')

print(f'\nTotal reports generated: {len(all_reports)}')
print('\nSample report (latest week):')
print(all_reports[-1]['report'])

Generating weekly reports...

Week 2024-08-05: Done
Week 2024-08-12: Done
Week 2024-08-19: Done
Week 2024-08-26: Done
Week 2024-09-02: Done
Week 2024-09-09: Done
Week 2024-09-16: Done
Week 2024-09-23: Done
Week 2024-09-30: Done

Total reports generated: 9

Sample report (latest week):
**Weekly Operational Report**

This week's waste composition data reveals a notable increase in paper-cardboard waste, with a significant surge of 817 annotations across 63 images. Additionally, an unusual spike in metal waste was detected on [2024-09-30], with 300 annotations significantly higher than the average daily amount. These trends suggest potential changes in the facility's waste stream, and further analysis is recommended to identify underlying causes. Notably, mixed waste and plastic waste volumes remained relatively stable compared to previous weeks.


## 7. Save Reports to Cloud SQL
Store generated weekly reports in llm_insights table for dashboard display and API serving.

In [7]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

cur.execute("DELETE FROM llm_insights WHERE insight_type = 'weekly_report'")

for r in all_reports:
    cur.execute("""
        INSERT INTO llm_insights (insight_date, insight_type, content)
        VALUES (%s, %s, %s)
    """, (str(r['week_start'])[:10], 'weekly_report', r['report']))

conn.commit()
cur.close()
conn.close()
print(f'Saved {len(all_reports)} weekly reports to Cloud SQL')

Saved 9 weekly reports to Cloud SQL


## 8. Prompt Engineering — Test and Refine
Test different prompt styles to find the most consistent
and accurate output format for operational reports.

In [8]:
# Test 3 prompt styles and compare output

test_context = build_weekly_context(df_weekly, df_anomaly,
                  week_start=df_weekly['week_start'].unique()[4])  # week with anomaly

# Prompt style 1 — descriptive
prompt1 = f"""You are an AI analyst for an industrial waste facility.
Summarize this weekly waste data in 3 sentences for facility managers.

Data:
{test_context}

Summary:"""

# Prompt style 2 — structured output
prompt2 = f"""Analyze the following waste composition data and respond in this exact format:
TREND: [one sentence about overall trend]
ANOMALY: [one sentence about any anomalies, or "None detected"]
ACTION: [one sentence operational recommendation]

Data:
{test_context}

Analysis:"""

# Prompt style 3 — bullet points
prompt3 = f"""Based on this waste facility data, provide:
- Key finding (1 sentence)
- Anomaly alert if any (1 sentence)
- Recommendation (1 sentence)

Data:
{test_context}

Report:"""

print('=== Prompt Style 1 — Descriptive ===')
print(ollama_generate(prompt1, max_tokens=200))
print('\n=== Prompt Style 2 — Structured ===')
print(ollama_generate(prompt2, max_tokens=200))
print('\n=== Prompt Style 3 — Bullet Points ===')
print(ollama_generate(prompt3, max_tokens=200))

=== Prompt Style 1 — Descriptive ===
Here is a summary of the weekly waste data in 3 sentences for facility managers:

This week, the total waste volume was slightly below average, with an anomaly detected on September 2nd where plastic and paper-cardboard volumes were unusually high. The facility also experienced anomalous days on September 5th, where plastic and paper-cardboard volumes exceeded their daily averages. Additionally, metal waste was significantly absent on September 3rd due to a complete absence in both the total volume and individual components.

=== Prompt Style 2 — Structured ===
TREND: The overall trend indicates a steady increase in waste composition with significant increases in plastic and paper-cardboard annotations across the week.

ANOMALY: None detected.

ACTION: Implement daily monitoring of specific materials to identify unusual patterns or fluctuations, such as Plastic and Paper-Cardboard, to ensure prompt action can be taken to adjust collection strategies

## 9. Select Best Prompt Style
Choose the most consistent prompt style based on output quality.

Document the decision for reproducibility.

In [9]:
# Selected prompt template — use this for all LLM calls going forward
WEEKLY_REPORT_PROMPT = """You are an AI analyst for an industrial waste incineration facility in Japan.
Based on the following waste composition data, respond in this exact format:
TREND: [one sentence about the dominant waste type and overall volume]
ANOMALY: [one sentence about anomalies detected, or "No anomalies detected this week"]
ACTION: [one sentence operational recommendation for facility managers]

Data:
{context}

Analysis:"""

# Test final prompt
test_report = ollama_generate(
    WEEKLY_REPORT_PROMPT.format(context=test_context),
    max_tokens=200
)
print('=== Final Prompt Output ===')
print(test_report)
print('\nPrompt template saved for use in all subsequent LLM calls')

=== Final Prompt Output ===
TREND: The dominant waste type this week is Paper-Cardboard, accounting for approximately 57% of the total waste volume, followed by Plastic at around 26%.
ANOMALY: No anomalies detected this week.
ACTION: Facility managers should closely monitor the high volumes of Plastic and Paper-Cardboard on days when their daily averages are exceeded to prevent potential operational issues with incineration equipment.

Prompt template saved for use in all subsequent LLM calls


## 10. Setup ChromaDB
Initialize ChromaDB as local vector store for storing operational documents and reports for RAG retrieval.

In [10]:
import chromadb
from chromadb.utils import embedding_functions

# Setup ChromaDB local persistent storage
CHROMA_PATH = '../data/chromadb'
client_chroma = chromadb.PersistentClient(path=CHROMA_PATH)

# Use default embedding function (sentence-transformers)
ef = embedding_functions.DefaultEmbeddingFunction()

# Create collection for waste documents
collection = client_chroma.get_or_create_collection(
    name='waste_documents',
    embedding_function=ef
)

print(f'ChromaDB initialized at: {CHROMA_PATH}')
print(f'Collection: waste_documents')
print(f'Documents in collection: {collection.count()}')

ChromaDB initialized at: ../data/chromadb
Collection: waste_documents
Documents in collection: 0


## 11. Prepare Documents for RAG
Convert weekly reports, anomaly explanations, and daily summaries into documents for embedding and storage in ChromaDB.

In [11]:
import uuid

documents  = []
metadatas  = []
ids        = []

# Add weekly reports as documents
for r in all_reports:
    doc_id = str(uuid.uuid4())
    documents.append(r['report'])
    metadatas.append({
        'type'      : 'weekly_report',
        'week_start': str(r['week_start'])[:10],
        'source'    : 'llm_generated'
    })
    ids.append(doc_id)

# Add anomaly explanations as documents
df_anomaly_all = query_db("""
    SELECT date, explanation, annotation_count, anomaly_score
    FROM anomaly_results
    WHERE is_anomaly = TRUE
""", DB_CONFIG)

for _, row in df_anomaly_all.iterrows():
    doc_id = str(uuid.uuid4())
    documents.append(f"Anomaly on {str(row['date'])[:10]}: {row['explanation']}")
    metadatas.append({
        'type'          : 'anomaly_event',
        'date'          : str(row['date'])[:10],
        'anomaly_score' : str(row['anomaly_score'])
    })
    ids.append(doc_id)

# Add data context summaries as documents
for week in df_weekly['week_start'].unique():
    ctx    = build_weekly_context(df_weekly, df_anomaly, week_start=week)
    doc_id = str(uuid.uuid4())
    documents.append(ctx)
    metadatas.append({
        'type'      : 'data_context',
        'week_start': str(week)[:10],
        'source'    : 'cloud_sql'
    })
    ids.append(doc_id)

print(f'Total documents prepared: {len(documents)}')
print(f'  Weekly reports    : {len(all_reports)}')
print(f'  Anomaly events    : {len(df_anomaly_all)}')
print(f'  Data contexts     : {len(df_weekly["week_start"].unique())}')

Total documents prepared: 22
  Weekly reports    : 9
  Anomaly events    : 4
  Data contexts     : 9


C:\Users\bhumi\AppData\Local\Temp\ipykernel_29616\752173889.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql(sql, conn)


## 12. Embed and Store Documents in ChromaDB
Convert all documents to vector embeddings and store in ChromaDB.

This enables semantic similarity search for RAG retrieval.

In [12]:
# Add all documents to ChromaDB
collection.add(
    documents=documents,
    metadatas=metadatas,
    ids=ids
)

print(f'Documents stored in ChromaDB: {collection.count()}')

# Test retrieval
test_query  = 'What happened on anomalous days?'
test_result = collection.query(query_texts=[test_query], n_results=2)

print(f'\nTest query: "{test_query}"')
print('Top 2 retrieved documents:')
for i, (doc, meta) in enumerate(zip(test_result['documents'][0],
                                     test_result['metadatas'][0])):
    print(f'\n  [{i+1}] Type: {meta["type"]}')
    print(f'       {doc[:150]}...')

C:\Users\bhumi\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:26<00:00, 3.19MiB/s]


Documents stored in ChromaDB: 22

Test query: "What happened on anomalous days?"
Top 2 retrieved documents:

  [1] Type: anomaly_event
       Anomaly on 2024-09-30: Metal was unusually high at 300 (avg: 45)...

  [2] Type: anomaly_event
       Anomaly on 2024-09-03: Total waste volume was 1724 annotations — more than 2x the daily average (637) | Paper-Cardboard was unusually high at 807 (avg...


## 13. RAG Pipeline - Query Function
Build full RAG pipeline: receive natural language query, retrieve relevant documents from ChromaDB, then generate answer using Ollama with retrieved context.

In [13]:
def rag_query(question, collection, n_results=3, model=OLLAMA_MODEL):
    """
    Full RAG pipeline:
    1. Retrieve relevant documents from ChromaDB
    2. Build context from retrieved docs
    3. Generate answer using Ollama
    """
    # Step 1 — Retrieve relevant documents
    results = collection.query(
        query_texts=[question],
        n_results=n_results
    )

    retrieved_docs  = results['documents'][0]
    retrieved_metas = results['metadatas'][0]

    # Step 2 — Build context from retrieved docs
    context_parts = []
    for doc, meta in zip(retrieved_docs, retrieved_metas):
        context_parts.append(f"[{meta['type']} — {meta.get('date', meta.get('week_start', ''))}]\n{doc}")
    context = '\n\n'.join(context_parts)

    # Step 3 — Generate answer
    prompt = f"""You are an AI analyst for an industrial waste incineration facility in Japan.
Answer the following question based only on the provided context.
If the answer is not in the context, say "I don't have enough data to answer this."
Be concise and professional.

Context:
{context}

Question: {question}

Answer:"""

    answer = ollama_generate(prompt, max_tokens=300)

    return {
        'question'      : question,
        'answer'        : answer,
        'retrieved_docs': len(retrieved_docs),
        'sources'       : [m['type'] for m in retrieved_metas]
    }

print('RAG pipeline defined successfully')

RAG pipeline defined successfully


## 14. Test RAG Pipeline with Sample Questions
Test the RAG pipeline with various operational questions to verify retrieval quality and answer accuracy.

In [14]:
test_questions = [
    "Which waste class had the highest volume this month?",
    "Were there any anomalous days in September?",
    "What happened on 2024-09-02?",
    "What is the operational recommendation for this week?",
]

print('=== RAG Pipeline Test ===\n')
for q in test_questions:
    result = rag_query(q, collection)
    print(f'Q: {result["question"]}')
    print(f'A: {result["answer"]}')
    print(f'Sources: {result["sources"]}')
    print()

=== RAG Pipeline Test ===

Q: Which waste class had the highest volume this month?
A: Based on the provided context, I don't have enough data to answer this question as only the specific days mentioned (2024-09-02 and 2024-09-05) are provided. The question seems to be asking for an overall answer for "this month", but without knowing which day of the month is being referred to or what the monthly average waste volumes are, I cannot provide a definitive answer.
Sources: ['anomaly_event', 'anomaly_event', 'weekly_report']

Q: Were there any anomalous days in September?
A: Yes, there was an anomalous day on September 30th, where metal detection exceeded the average by 300 (avg: 45).
Sources: ['anomaly_event', 'data_context', 'data_context']

Q: What happened on 2024-09-02?
A: On 2024-09-02, the total waste volume was over 2x the daily average, and there were anomalies in several categories: Plastic was unusually high at 1337, Paper-Cardboard was unusually high at 1168, Mixed Waste was unu

## 15. Evaluate RAG Retrieval Quality
Check if retrieved documents are relevant to each query by inspecting source types and content.

In [15]:
print('=== Retrieval Quality Check ===\n')

eval_queries = [
    ('anomaly query',  'What anomalies were detected?'),
    ('trend query',    'What is the trend for Plastic waste?'),
    ('report query',   'Summarize the latest weekly report'),
]

for query_type, question in eval_queries:
    results = collection.query(query_texts=[question], n_results=3)
    sources = [m['type'] for m in results['metadatas'][0]]
    print(f'{query_type:<20} → sources retrieved: {sources}')

=== Retrieval Quality Check ===

anomaly query        → sources retrieved: ['anomaly_event', 'anomaly_event', 'anomaly_event']
trend query          → sources retrieved: ['anomaly_event', 'weekly_report', 'weekly_report']
report query         → sources retrieved: ['data_context', 'data_context', 'data_context']


## 16. Anomaly Explanation Module
Generate detailed business explanations for each anomaly detected by Isolation Forest in Phase 4.

Explanations are written for non-technical stakeholders.

In [16]:
def explain_anomaly_llm(anomaly_row, model=OLLAMA_MODEL):
    """
    Generate LLM-powered business explanation for a single anomaly day.
    Takes a row from anomaly_results table.
    """
    prompt = f"""You are an operations analyst at an industrial waste incineration facility in Japan.
A statistical anomaly was detected on {str(anomaly_row['date'])[:10]}.

Anomaly details:
- Total annotations: {int(anomaly_row['annotation_count'])}
- Anomaly score: {float(anomaly_row['anomaly_score']):.4f} (more negative = more anomalous)
- Statistical finding: {anomaly_row['explanation']}

Write a 2-sentence operational explanation for facility managers that:
1. Describes what happened in plain language
2. Suggests a possible operational cause or recommended action

Explanation:"""

    return ollama_generate(prompt, max_tokens=150)

# Generate explanations for all anomaly days
df_anomaly_full = query_db("""
    SELECT date, is_anomaly, anomaly_score, annotation_count, explanation
    FROM anomaly_results
    WHERE is_anomaly = TRUE
    ORDER BY anomaly_score
""", DB_CONFIG)

print('=== Anomaly Explanation Module ===\n')
anomaly_explanations = []

for _, row in df_anomaly_full.iterrows():
    explanation = explain_anomaly_llm(row)
    anomaly_explanations.append({
        'date'       : str(row['date'])[:10],
        'explanation': explanation
    })
    print(f"Date: {str(row['date'])[:10]}")
    print(f"Explanation: {explanation}")
    print()

C:\Users\bhumi\AppData\Local\Temp\ipykernel_29616\752173889.py:14: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df   = pd.read_sql(sql, conn)


=== Anomaly Explanation Module ===

Date: 2024-09-02
Explanation: On September 2nd, our facility experienced an unexpected surge in waste volume and composition, with unusually high levels of plastic, paper-cardboard, mixed waste, and metal compared to our daily averages, which may have been caused by the unusually high number of construction projects taking place in the surrounding areas.

We recommend that we monitor this trend closely and consider increasing our waste processing capacity or modifying our waste segregation protocols to ensure that we can effectively handle the increased volume and composition of waste, while also maintaining our environmental standards and safety protocols.

Date: 2024-09-05
Explanation: On September 5th, our incineration facility experienced an unusual spike in waste volume, particularly plastics and papers, which exceeded their daily averages by more than two times. We recommend inspecting the facility's collection procedures to ensure that the unu

## 17. Save Anomaly Explanations to Cloud SQL
Store LLM-generated anomaly explanations in llm_insights table for dashboard alerts and API serving.

In [17]:
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

cur.execute("DELETE FROM llm_insights WHERE insight_type = 'anomaly_explanation'")

for item in anomaly_explanations:
    cur.execute("""
        INSERT INTO llm_insights (insight_date, insight_type, content)
        VALUES (%s, %s, %s)
    """, (item['date'], 'anomaly_explanation', item['explanation']))

conn.commit()
cur.close()
conn.close()
print(f'Saved {len(anomaly_explanations)} anomaly explanations to Cloud SQL')

Saved 4 anomaly explanations to Cloud SQL


## 18. Finalize LLM Module - Summary
Verify all LLM components are working correctly and all results are stored in Cloud SQL.

In [18]:
# Final verification
conn = psycopg2.connect(**DB_CONFIG)
cur  = conn.cursor()

cur.execute("SELECT insight_type, COUNT(*) FROM llm_insights GROUP BY insight_type")
rows = cur.fetchall()

cur.close()
conn.close()

print('=== LLM Module Summary ===\n')
print('Cloud SQL llm_insights table:')
for row in rows:
    print(f'  {row[0]:<25} {row[1]} records')

print('\nComponents completed:')
print('  ✓ Data-to-Text pipeline     — weekly reports generated')
print('  ✓ Prompt engineering        — structured TREND/ANOMALY/ACTION format')
print('  ✓ ChromaDB vector store     — 22 documents embedded')
print('  ✓ RAG pipeline              — natural language querying')
print('  ✓ Anomaly explanation       — LLM explanations for all anomaly days')
print('  ✓ Cloud SQL storage         — all insights saved')

=== LLM Module Summary ===

Cloud SQL llm_insights table:
  weekly_report             9 records
  anomaly_explanation       4 records

Components completed:
  ✓ Data-to-Text pipeline     — weekly reports generated
  ✓ Prompt engineering        — structured TREND/ANOMALY/ACTION format
  ✓ ChromaDB vector store     — 22 documents embedded
  ✓ RAG pipeline              — natural language querying
  ✓ Anomaly explanation       — LLM explanations for all anomaly days
  ✓ Cloud SQL storage         — all insights saved
